# 掩码语言模型训练实例

## Step1 导入相关包

In [1]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer

## Step2 加载数据集

In [2]:
ds = Dataset.load_from_disk("./wiki_cn_filtered/")

In [3]:
ds

Dataset({
    features: ['source', 'completion'],
    num_rows: 10000
})

In [4]:
ds[0]

{'source': 'wikipedia.zh2307',
 'completion': "西安交通大学博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆，馆长是锺明善。\n历史\n2004年9月20日开始筹建，2013年4月8日正式建成开馆，位于西安交通大学兴庆校区陕西省西安市咸宁西路28号。建筑面积6,800平米，展厅面积4,500平米，馆藏文物4,900余件。包括历代艺术文物馆、碑石书法馆、西部农民画馆、邢良坤陶瓷艺术馆、陕西秦腔博物馆和书画展厅共五馆一厅。\n营业时间\n* 周一至周六：上午九点至十二点，下午一点至五点\n* 周日闭馆"}

## Step3 数据集处理

In [5]:
tokenizer = AutoTokenizer.from_pretrained("../../model/hfl/chinese-macbert-base")

def process_func(examples):
    return tokenizer(examples["completion"], max_length=384, truncation=True)

In [6]:
tokenized_ds = ds.map(process_func, batched=True, remove_columns=ds.column_names)
tokenized_ds

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})

In [7]:
# from torch.utils.data import DataLoader

# dl = DataLoader(tokenized_ds, batch_size=2, collate_fn=DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15))

In [8]:
# next(enumerate(dl))

In [9]:
tokenizer.mask_token, tokenizer.mask_token_id

('[MASK]', 103)

## Step4 创建模型

In [10]:
model = AutoModelForMaskedLM.from_pretrained("../../model/hfl/chinese-macbert-base")

Some weights of the model checkpoint at ../../model/hfl/chinese-macbert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


## Step5 配置训练参数

In [11]:
args = TrainingArguments(
    output_dir="./masked_lm",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    logging_steps=10,
    num_train_epochs=1
)

## Step6 创建训练器

In [12]:
trainer = Trainer(
    args=args,
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15)
)

## Step7 模型训练

In [13]:
trainer.train()

  0%|          | 0/625 [00:00<?, ?it/s]

{'loss': 1.4163, 'grad_norm': 5.016941547393799, 'learning_rate': 4.92e-05, 'epoch': 0.02}
{'loss': 1.3954, 'grad_norm': 4.824465274810791, 'learning_rate': 4.8400000000000004e-05, 'epoch': 0.03}
{'loss': 1.4511, 'grad_norm': 5.139054775238037, 'learning_rate': 4.76e-05, 'epoch': 0.05}
{'loss': 1.3536, 'grad_norm': 4.414644718170166, 'learning_rate': 4.6800000000000006e-05, 'epoch': 0.06}
{'loss': 1.4179, 'grad_norm': 4.841177463531494, 'learning_rate': 4.600000000000001e-05, 'epoch': 0.08}
{'loss': 1.3911, 'grad_norm': 4.5393900871276855, 'learning_rate': 4.52e-05, 'epoch': 0.1}
{'loss': 1.4099, 'grad_norm': 5.263214588165283, 'learning_rate': 4.44e-05, 'epoch': 0.11}
{'loss': 1.3215, 'grad_norm': 4.330409049987793, 'learning_rate': 4.36e-05, 'epoch': 0.13}
{'loss': 1.3865, 'grad_norm': 5.153807163238525, 'learning_rate': 4.2800000000000004e-05, 'epoch': 0.14}
{'loss': 1.438, 'grad_norm': 4.863065242767334, 'learning_rate': 4.2e-05, 'epoch': 0.16}
{'loss': 1.424, 'grad_norm': 4.407543

TrainOutput(global_step=625, training_loss=1.3376720764160157, metrics={'train_runtime': 544.7022, 'train_samples_per_second': 18.359, 'train_steps_per_second': 1.147, 'total_flos': 1973194615348224.0, 'train_loss': 1.3376720764160157, 'epoch': 1.0})

## Step8 模型推理

In [14]:
from transformers import pipeline

pipe = pipeline("fill-mask", model=model, tokenizer=tokenizer, device=0)

In [15]:
pipe("西安交通[MASK][MASK]博物馆（Xi'an Jiaotong University Museum）是一座位于西安交通大学的博物馆")

[[{'score': 0.9993525147438049,
   'token': 1920,
   'token_str': '大',
   'sequence': "[CLS] 西 安 交 通 大 [MASK] 博 物 馆 （ xi'an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 0.00046250942978076637,
   'token': 2110,
   'token_str': '学',
   'sequence': "[CLS] 西 安 交 通 学 [MASK] 博 物 馆 （ xi'an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 3.912094689439982e-05,
   'token': 4906,
   'token_str': '科',
   'sequence': "[CLS] 西 安 交 通 科 [MASK] 博 物 馆 （ xi'an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 2.8664902856689878e-05,
   'token': 5466,
   'token_str': '职',
   'sequence': "[CLS] 西 安 交 通 职 [MASK] 博 物 馆 （ xi'an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"},
  {'score': 1.8002236174652353e-05,
   'token': 2339,
   'token_str': '工',
   'sequence': "[CLS] 西 安 交 通 工 [MASK] 博 物 馆 （ xi'an jiaotong university museum ） 是 一 座 位 于 西 安 交 通 大 学 的 博 物 馆 [SEP]"}],
 [{'score': 0.9960961937

In [16]:
pipe("下面是一则[MASK][MASK]新闻。小编报道，近日，游戏产业发展的非常好！")

[[{'score': 0.07796819508075714,
   'token': 2031,
   'token_str': '娱',
   'sequence': '[CLS] 下 面 是 一 则 娱 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.05413195490837097,
   'token': 4685,
   'token_str': '相',
   'sequence': '[CLS] 下 面 是 一 则 相 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.03692091256380081,
   'token': 5381,
   'token_str': '网',
   'sequence': '[CLS] 下 面 是 一 则 网 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.035038892179727554,
   'token': 4178,
   'token_str': '热',
   'sequence': '[CLS] 下 面 是 一 则 热 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
  {'score': 0.031335119158029556,
   'token': 3952,
   'token_str': '游',
   'sequence': '[CLS] 下 面 是 一 则 游 [MASK] 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'}],
 [{'score': 0.07551957666873932,
   'token': 6380,
   'token_str': '讯',
   'sequence': '[CLS] 下 面 是 一 则 [MASK] 讯 新 闻 。 小 编 报 道 ， 近 日 ， 游 戏 产 业 发 展 的 非 常 好 ！ [SEP]'},
 